# Klasifikasi DemogPairs Menggunakan ViT (Umur) & Gaussian Naive Bayes

In [1]:
import numpy as np
import utils as u
import joblib
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from tqdm import tqdm

joblib.parallel_backend('threading')

## Load Dataset

In [3]:
data = u.load_demogpairs()
pd.DataFrame(data)

## Load Fitur

In [5]:
features = joblib.load('features/demogpairs_vit-age.pkl')
print('Jumlah fitur per gambar:', np.array(features[list(features.keys())[0]]).shape[0])

Jumlah fitur per gambar: 768


## Split Data

In [7]:
X = np.array([features[d['image_path']] for d in data])
y = np.array([d['label_idx'] for d in data])
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
print((len(X_train), len(X_test)))

(8640, 2160)


## Kombinasi Parameter

In [9]:
var_smoothing_values = np.logspace(-9, 2, 40)  # dari 1e-9 sampai 1e2, 40 nilai

grid_params = [
    {
        'scaler': [None, MinMaxScaler()],
        'pca': [None, PCA(n_components=0.5), PCA(n_components=0.75)],
        
        'classifier': [GaussianNB()],
        'classifier__var_smoothing': var_smoothing_values
    },
]

pipeline = Pipeline(steps=[
    ('scaler', None),
    ('pca', None),
    ('classifier', None)
])

skv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy', 
    'f1': 'f1_macro', 
    'precision': 'precision_macro', 
    'recall': 'recall_macro',
    'roc_auc_ovr': 'roc_auc_ovr'
}

grid_models = {}
for params in grid_params:
    key = str(params['classifier'][0]).split('(')[0]
    grid_models[key] = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        cv=skv, refit='accuracy',
        scoring=scoring, n_jobs=int(joblib.cpu_count() * 0.6),
        verbose=1, error_score='raise',
        return_train_score=True
    )
    print(f'{key}: {len(ParameterGrid(params))} kombinasi')

GaussianNB: 240 kombinasi


## Klasifikasi

In [11]:
evaluation_results, fold_results = u.evaluate_models(
    grid_models,
    X_train, y_train,
    X_test, y_test,
    model_prefix='models/clf_demogpairs_gnb_vit-age_',
    results_path='results/demogpairs_gnb_vit-age_'
)

sorted_results = pd.DataFrame(evaluation_results).sort_values(by='test_accuracy', ascending=False).to_dict('records')
u.html_br()
_dtable = u.display_table(sorted_results)

Evaluating: GaussianNB

##### Best Parameters
{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.000437547937507418), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}

##### Test Results
Accuracy  : 0.6962962962962963
Precision : 0.697890477865101
Recall    : 0.6962962962962962
F1 Score  : 0.6951754820058508
              precision    recall  f1-score   support

           0       0.65      0.70      0.67       360
           1       0.69      0.75      0.72       360
           2       0.69      0.61      0.65       360
           3       0.70      0.78      0.74       360
           4       0.74      0.63      0.68       360
           5       0.71      0.71      0.71       360

    accuracy                           0.70      2160
   macro avg       0.70      0.70      0.70      2160
weighted avg       0.70      0.70      0.70      2160


Class
    Accuracy
    Precision
    Recall
    F1-Score
  
  
    Black_Males
    0.8875
    0.6511627906976745
    0.7
    0.67469879

In [12]:
model, training_time = u.load_object('models/clf_demogpairs_gnb_vit-age_GaussianNB.pkl')
u.h(5, 'Waktu Pelatihan (Jobs)')
u.seconds_to_time(round(training_time))


##### Waktu Pelatihan (Jobs)


In [13]:
u.h(5, 'Waktu Pelatihan')
times = [fr['Train Time Mean'] * 5 for fr in fold_results]
u.seconds_to_time(round(np.sum(times) + model.refit_time_))


##### Waktu Pelatihan


In [14]:
_dtable = u.display_table(fold_results, n_items=[4, 4], column_widths=['5%', '45%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%'])

No
    Params
    Fold 1
    Fold 2
    Fold 3
    Fold 4
    Fold 5
    Accuracy Mean
    F1 Score Mean
    Precision Mean
    Recall Mean
    Train Time Mean
  
  
    1
    {'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.000437547937507418), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}
    0.7002
    0.6817
    0.6875
    0.6927
    0.6973
    0.6919
    0.6902
    0.694
    0.6919
    2.5738
  
  
    2
    {'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.00022854638641349884), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}
    0.6979
    0.6823
    0.6858
    0.6927
    0.6968
    0.6911
    0.6896
    0.6929
    0.6911
    3.0914
  
  
    3
    {'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(3.257020655659783e-05), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}
    0.6991
    0.6817
    0.6834
    0.6944
    0.6927
    0.6903
    0.6889
    0.692
    0.6903
    3.1135
  
  
    4
    {'classifier': 'GaussianNB', 'classifier__var_smo